## Import libraries


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import math

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

## Read the dataset


In [ ]:
# Read ROI
roi_gdf = gpd.read_file(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/raw/shapefile/west_bengal.gpkg"
)

# Read the crop samples
crop_samples = gpd.read_file(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/master/wbcrop_points.gpkg"
)
crop_samples = crop_samples.to_crs(crs="EPSG:32645")

## Create panel map


In [ ]:
# Crop colors
crop_colors = {
    "aman_rice": "#1f77b4",
    "aus_rice": "#ff7f0e",
    "banana": "#2ca02c",
    "betel_leaf": "#d62728",
    "boro_rice": "#9467bd",
    "flower": "#8c564b",
    "groundnut": "#e377c2",
    "jute": "#7f7f7f",
    "maize": "#bcbd22",
    "mustard": "#17becf",
    "others": "#aec7e8",
    "pine_apple": "#ffbb78",
    "potato": "#98df8a",
    "sugarcane": "#ff9896",
    "tea": "#c5b0d5",
    "tobacco": "#c49c94",
    "vegetables": "#f7b6d2",
    "wheat": "#dbdb8d",
}

In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Assuming crop_colors and crop_samples are already loaded in your environment

# Path to high-res patches
patches_dir = (
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/raw/high_res_patches/high_res_patches"
)

# Get all patch files
patch_files = [f for f in os.listdir(patches_dir) if f.endswith(".tif")]

# Extract patch IDs and crop types
patch_info = []
for patch_file in patch_files:
    parts = patch_file.replace(".tif", "").split("_", 1)
    if len(parts) == 2:
        patch_id, crop_type = parts
        patch_info.append(
            {"filename": patch_file, "patch_id": patch_id, "crop_type": crop_type}
        )

print(f"Total patches: {len(patch_info)}")
print(f"Unique crops: {len(set(p['crop_type'] for p in patch_info))}")

# Create a dictionary to select one representative patch for each crop
crop_to_patch = {}
for patch_dict in patch_info:
    crop_type = patch_dict["crop_type"]
    if crop_type not in crop_to_patch:
        crop_to_patch[crop_type] = patch_dict

# Sort crops to match the order in crop_colors
crop_order = list(crop_colors.keys())
selected_patches = [crop_to_patch[crop] for crop in crop_order if crop in crop_to_patch]

print(f"Selected {len(selected_patches)} crops for panel visualization")

n_patches = len(selected_patches)
ncols = 4
nrows = 5  # Fixed for 18 crops

# Use constrained_layout for perfect spacing between titles and images
fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols, figsize=(8, 11), constrained_layout=True
)
axes = axes.flatten()

for idx, patch_dict in enumerate(selected_patches):
    ax = axes[idx]
    patch_path = os.path.join(patches_dir, patch_dict["filename"])
    patch_id = patch_dict["patch_id"]
    crop_type = patch_dict["crop_type"]

    crop_label = crop_type.replace("_", " ").title()

    # Read the patch
    try:
        with rasterio.open(patch_path) as src:
            patch_array = src.read()

            # Normalize for visualization (Percentile Stretch for better contrast)
            if len(patch_array.shape) == 3 and patch_array.shape[0] >= 3:
                rgb = patch_array[:3].astype(float)
                p2, p98 = np.percentile(rgb, (2, 98))
                rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
                rgb = np.moveaxis(rgb, 0, -1)
                ax.imshow(rgb)
            else:
                band = patch_array[0] if len(patch_array.shape) == 3 else patch_array
                ax.imshow(band, cmap="viridis")

            # Get corresponding crop point from crop_samples
            point_in_patch = crop_samples[crop_samples["id"].astype(str) == patch_id]

            if not point_in_patch.empty:
                point = point_in_patch.iloc[0].geometry
                lat = point_in_patch.iloc[0]["latitude"]
                lon = point_in_patch.iloc[0]["longitude"]

                # Use Rasterio's built-in spatial indexer for exact row/col mapping
                py, px = src.index(point.x, point.y)

                # Plot point with crop color
                color = crop_colors.get(crop_type, "#000000")
                ax.plot(
                    px,
                    py,
                    marker="o",
                    color=color,
                    markersize=14,
                    markeredgecolor="white",
                    markeredgewidth=2,
                    zorder=5,
                )

                # Set Title with Crop Name and Coordinates
                # point.y is usually Latitude, point.x is usually Longitude
                title_text = f"{crop_label}\nLat: {lat:.2f}, Lon: {lon:.2f}"
                ax.set_title(title_text, fontsize=9, fontweight="bold")
            else:
                # Fallback title if point isn't found
                ax.set_title(crop_label, fontsize=12)

    except Exception as e:
        print(f"Error loading {patch_dict['filename']}: {str(e)}")
        ax.set_title(crop_label, fontsize=12, fontweight="bold")

    # Add thick black frame around the image
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1)
        spine.set_visible(True)

    # Remove internal image axes ticks
    ax.set_xticks([])
    ax.set_yticks([])

# Remove empty subplots if any (handles cases where you might have fewer than 18 crops found)
for j in range(n_patches, len(axes)):
    fig.delaxes(axes[j])

# Ensure output directory exists before saving
output_path = "/beegfs/halder/GITHUB/RESEARCH/WBCrop/output/figures/panel_highres_patches_18crops.png"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

plt.subplots_adjust(hspace=0.8)
plt.savefig(output_path, dpi=96, bbox_inches="tight")
plt.show()

print(f"Panel map with {n_patches} crops created and saved!")